# LangChain Agent vs. Mellea Agent: Planning a Trip, Two Ways

An **agent** lets a model decide, turn by turn, which tool to call and when to stop. This recipe builds the same vacation-planning agent twice, sharing one LangChain tool object between both:

- [LangChain](https://www.langchain.com/)'s `create_agent` (built on LangGraph)
- [Mellea](https://mellea.ai)'s `react()`, wrapping the identical `destination_guide` tool via `MelleaTool.from_langchain()`, zero rewrite

Both check candidates against a budget and season, then return a typed itinerary, and both get a budget check enforced, one with a hand-rolled retry, the other with Mellea's own repair-with-reason loop.

**What you'll build:**

- `destination_guide`: a small in-memory dataset of destinations (cost, best months, highlights).
- A LangChain agent: `ChatOllama` + `create_agent` + `ToolCallLimitMiddleware`, returning a typed itinerary.
- The same tool, reused unmodified in Mellea via `MelleaTool.from_langchain()`.
- A Mellea agent on the built-in `react()` loop, returning the same typed `VacationPlan`.
- A budget check enforced two ways: a hand-rolled retry for LangChain, a real `Requirement` + `RepairTemplateStrategy` for Mellea.
- A side-by-side comparison: setup, tool definition, output typing, loop control, validation, observability.

Runs on a single local Ollama model throughout, not backend flexibility. For watsonx.ai, see [Quick Start](../QuickStart/QuickStart.ipynb).

## Step 1. Install relevant libraries

- **`mellea`**: `start_session`, `react()`, `MelleaTool.from_langchain()`, no extras needed.
- **`langchain` / `langchain-ollama`**: `create_agent`, the `@tool` decorator, `ChatOllama`.
- **`pydantic`**: the typed `VacationPlan` schema.

No live web dependency: `destination_guide` is a plain function over an in-memory dataset, nothing to rate-limit.

In [ ]:
%pip install -q mellea langchain langchain-ollama pydantic

## Step 2. Set up a local Ollama backend

One Ollama daemon, one pulled model, both agents call it: LangChain's `ChatOllama` and Mellea's native `ollama` backend both hit the same local API.

Run this cell to install Ollama and pull the model. Already have Ollama running locally with the model pulled? Skip it.

In [ ]:
# Install Ollama
! sudo apt-get install -y zstd
! curl -fsSL https://ollama.com/install.sh | sh

# Start the Ollama daemon in the background
import subprocess
import time
subprocess.Popen(["ollama", "serve"])
time.sleep(5)

# Pull the IBM Granite model used by both agents below
! ollama pull granite4.2:3b

## Step 3. Import libraries and set constants

- One model tag builds both `ChatOllama` and the Mellea session: same weights, both agents.
- `TOOL_CALL_BUDGET` caps the loop for both frameworks.
- `TRIP_DAYS` / `DAILY_BUDGET_USD` describe the traveler; both agents get checked against the same numbers in Step 5.

In [ ]:
import pydantic
from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware
from langchain_core.tools import tool
from langchain_ollama import ChatOllama

from mellea import MelleaSession, start_session
from mellea.backends.model_options import ModelOption
from mellea.backends.tools import MelleaTool
from mellea.core import Requirement, ValidationResult
from mellea.stdlib.context import ChatContext
from mellea.stdlib.frameworks.react import react
from mellea.stdlib.sampling import RepairTemplateStrategy

OLLAMA_MODEL_TAG = "granite4.2:3b"   # shared by ChatOllama (LangChain) and the Mellea session
TOOL_CALL_BUDGET = 6                  # loop control cap, applied to both agents below
MAX_NEW_TOKENS = 1024                 # output-length cap, both agents: bounds a runaway generation

TRIP_DAYS = 4
DAILY_BUDGET_USD = 120

vacation_request = (
    f"Plan a {TRIP_DAYS}-day vacation for a traveler with a budget of "
    f"${DAILY_BUDGET_USD} per day who wants warm weather, loves nature and local "
    "culture, and is traveling in April. Recommend one destination and give a "
    "day-by-day itinerary with an estimated total cost."
)

print("Model:", OLLAMA_MODEL_TAG)
print("Request:", vacation_request)

## Step 4. The shared tool: one LangChain object, both agents

- `@tool` turns a plain function into a LangChain `BaseTool`, reading its docstring and type hints.
- `destination_guide` looks up cost, best months, climate, and highlights for six destinations, enough to compare candidates before committing.
- Fully local and deterministic: no API key, no network call.

`destination_tool` is built once: used directly by LangChain's agent, then handed to Mellea's `MelleaTool.from_langchain()` unmodified.

*Mellea guide: [Tools and Agents: Defining tools with `@tool`](https://github.com/generative-computing/mellea/blob/main/docs/docs/how-to/tools-and-agents.md#defining-tools-with-tool)*

In [ ]:
DESTINATIONS = {
    "Bali, Indonesia": {
        "avg_daily_cost_usd": 65,
        "best_months": ["Apr", "May", "Jun", "Sep"],
        "climate": "tropical",
        "highlights": ["beaches", "rice terraces", "temples"],
    },
    "Costa Rica": {
        "avg_daily_cost_usd": 95,
        "best_months": ["Dec", "Jan", "Feb", "Mar"],
        "climate": "tropical",
        "highlights": ["rainforest hikes", "wildlife", "beaches"],
    },
    "Marrakech, Morocco": {
        "avg_daily_cost_usd": 70,
        "best_months": ["Mar", "Apr", "Oct", "Nov"],
        "climate": "arid",
        "highlights": ["markets", "desert day trips", "architecture"],
    },
    "Reykjavik, Iceland": {
        "avg_daily_cost_usd": 210,
        "best_months": ["Jun", "Jul", "Aug"],
        "climate": "subarctic",
        "highlights": ["waterfalls", "hot springs", "northern lights"],
    },
    "Tokyo, Japan": {
        "avg_daily_cost_usd": 150,
        "best_months": ["Mar", "Apr", "Oct", "Nov"],
        "climate": "temperate",
        "highlights": ["temples", "food markets", "gardens"],
    },
    "Lisbon, Portugal": {
        "avg_daily_cost_usd": 100,
        "best_months": ["Apr", "May", "Jun", "Sep"],
        "climate": "temperate",
        "highlights": ["old town", "coastal day trips", "local cuisine"],
    },
}


@tool
def destination_guide(destination: str) -> dict:
    """Look up travel-planning facts for a destination.

    Args:
        destination: Destination name, e.g. "Lisbon, Portugal".
    """
    needle = destination.strip().lower()
    for name, facts in DESTINATIONS.items():
        if needle in name.lower() or name.lower() in needle:
            return {"destination": name, **facts}
    return {"error": f"No data for '{destination}'.", "known_destinations": list(DESTINATIONS)}


destination_tool = destination_guide

print("Tool name:", destination_tool.name)
print("Tool description:", destination_tool.description)

## Step 5. A typed itinerary schema, and a validator for it

- Both agents return this schema instead of free text: `destination`, day-by-day `itinerary`, `estimated_total_cost_usd`.
- A schema guarantees *shape*, not correct numbers. `budget_check` is a plain, deterministic function (day count and cost fit `DAILY_BUDGET_USD * TRIP_DAYS`), reused by both agents below.
- `run_with_validated_retry` is `async`. Jupyter/Colab support top-level `await`, so every call below is `await run_with_validated_retry(...)`; `asyncio.run()` fails inside a notebook kernel's own event loop.

In [ ]:
class DayPlan(pydantic.BaseModel):
    day: int = pydantic.Field(description="Day number, starting at 1")
    activities: list[str] = pydantic.Field(description="Planned activities for this day")


class VacationPlan(pydantic.BaseModel):
    destination: str = pydantic.Field(description="The recommended destination")
    itinerary: list[DayPlan] = pydantic.Field(description="One entry per day of the trip")
    estimated_total_cost_usd: float = pydantic.Field(description="Estimated total cost in USD for the whole trip")


def budget_check(plan: VacationPlan) -> tuple[bool, str]:
    """Deterministic check: the itinerary covers every day and fits the budget."""
    if len(plan.itinerary) != TRIP_DAYS:
        return False, f"Itinerary has {len(plan.itinerary)} days; expected {TRIP_DAYS}."
    max_cost = DAILY_BUDGET_USD * TRIP_DAYS
    if plan.estimated_total_cost_usd > max_cost:
        return False, f"Estimated cost ${plan.estimated_total_cost_usd:.0f} exceeds the ${max_cost:.0f} budget."
    return True, f"{TRIP_DAYS}-day itinerary within the ${max_cost:.0f} budget."


async def run_with_validated_retry(run_once, goal: str, max_attempts: int = 3) -> VacationPlan:
    """Shared retry wrapper: call `run_once(goal)`, check it, retry on failure."""
    for attempt in range(1, max_attempts + 1):
        candidate = await run_once(goal)
        ok, reason = budget_check(candidate)
        print(f"  Attempt {attempt}: {'PASS' if ok else 'FAIL'} - {reason}")
        if ok:
            return candidate
    raise RuntimeError(f"No valid plan after {max_attempts} attempts.")


print("VacationPlan schema, budget_check, and run_with_validated_retry defined.")

## Step 6. LangChain agent: setup and tool-calling loop

- `create_agent` compiles a state graph: call the model, route to tools when requested, loop until it stops.
- Loop control is opt-in via middleware, here `ToolCallLimitMiddleware` capping tool calls per run.
- `create_agent` also accepts `response_format=` for typed output directly, left off here (see the caveat in Step 7).
- `temperature=0` would make decoding fully deterministic (same token every time). Left unset here (Ollama's default) instead: Step 8's retry reuses this same prompt, and a deterministic model would just repeat the same wrong answer on every attempt.

In [ ]:
llm = ChatOllama(model=OLLAMA_MODEL_TAG, num_predict=MAX_NEW_TOKENS)

langchain_agent = create_agent(
    model=llm,
    tools=[destination_tool],
    system_prompt=(
        "You are a vacation-planning assistant. Use the destination_guide tool to "
        "check a couple of candidate destinations against the traveler's budget, "
        "season, and interests before deciding. Recommend exactly one destination "
        "and produce a day-by-day itinerary with a realistic total cost estimate."
    ),
    middleware=[ToolCallLimitMiddleware(run_limit=TOOL_CALL_BUDGET, exit_behavior="end")],
)

print("LangChain agent ready.")

## Step 7. Run the LangChain agent, then extract typed output

- `response_format=` needs forced tool choice (`tool_choice="any"`). [`ChatOllama.bind_tools` ignores `tool_choice`](https://reference.langchain.com/python/langchain-ollama/chat_models/ChatOllama/bind_tools), since Ollama doesn't support it. Confirmed against `granite4.2:3b`: `response_format=VacationPlan` silently leaves `structured_response` as `None`.
- The reliable local-model pattern: two calls. Run the loop for a plain-text plan, then `llm.with_structured_output(VacationPlan)` to type it (native constrained decoding, no tool choice).

In [ ]:
structured_llm = llm.with_structured_output(VacationPlan)

langchain_result = langchain_agent.invoke(
    {"messages": [{"role": "user", "content": vacation_request}]}
)
final_text = langchain_result["messages"][-1].content

langchain_plan: VacationPlan = structured_llm.invoke(
    f"Extract the destination, day-by-day itinerary, and estimated total cost from this "
    f"vacation plan:\n\n{final_text}"
)

print("=== LANGCHAIN AGENT PLAN ===")
print(langchain_plan.model_dump_json(indent=2))
print("\nMessages exchanged in the tool-calling loop:", len(langchain_result["messages"]))

## Step 8. Validating the LangChain agent's plan

Typing only guarantees *shape*, not that the itinerary covers every day or that the cost estimate fits the budget, and nothing about `create_agent` or `with_structured_output` retries on bad numbers. This wraps the two calls from Step 7 in `run_with_validated_retry`, reusing `budget_check` from Step 5.

In [ ]:
async def run_langchain_agent(goal: str) -> VacationPlan:
    result = langchain_agent.invoke({"messages": [{"role": "user", "content": goal}]})
    final_text = result["messages"][-1].content
    return structured_llm.invoke(
        f"Extract the destination, day-by-day itinerary, and estimated total cost from "
        f"this vacation plan:\n\n{final_text}"
    )


print("=== VALIDATING LANGCHAIN AGENT ===")
validated_langchain_plan = await run_with_validated_retry(
    run_langchain_agent, vacation_request
)
print(validated_langchain_plan.model_dump_json(indent=2))

## Step 9. Reuse the same tool in Mellea, zero rewrite

Mellea has no destination-lookup tool of its own. `MelleaTool.from_langchain()` wraps the identical `BaseTool` the LangChain agent just used, remapping calls onto `tool.run(tool_input=...)` internally; `destination_tool` itself doesn't change.

*Mellea guide: [Tools and Agents: LangChain and smolagents interop](https://github.com/generative-computing/mellea/blob/main/docs/docs/how-to/tools-and-agents.md#langchain-and-smolagents-interop)*

In [ ]:
mellea_destination_tool: MelleaTool = MelleaTool.from_langchain(destination_tool)

print("Wrapped tool name:", mellea_destination_tool.name)

# Runs through the exact same destination_guide instance the LangChain agent used above.
direct_call = mellea_destination_tool.run(destination="Lisbon, Portugal")
print("Direct call returned:", direct_call)

## Step 10. Mellea agent: setup

- Start a session, then define `run_mellea_agent()`, a wrapper around `react()`: Mellea's built-in ReACT loop (think, pick a tool, observe, repeat, until `final_answer` is called or `loop_budget` runs out).
- Inside it, `format=VacationPlan` synthesizes the loop's last step into that schema instead of a raw string: one call, same typed-output guarantee as LangChain's `response_format`.
- This cell only defines the function; Step 11 calls it.

*Mellea guide: [Tools and Agents: ReACT agent](https://github.com/generative-computing/mellea/blob/main/docs/docs/how-to/tools-and-agents.md#react-agent)*

In [ ]:
m: MelleaSession = start_session(
    backend_name="ollama",
    model_id=OLLAMA_MODEL_TAG,
    ctx=ChatContext(),
    model_options={ModelOption.MAX_NEW_TOKENS: MAX_NEW_TOKENS},
)


async def run_mellea_agent(goal: str) -> VacationPlan:
    result, _ctx = await react(
        goal=goal,
        context=ChatContext(),
        backend=m.backend,
        tools=[mellea_destination_tool],
        format=VacationPlan,
        loop_budget=TOOL_CALL_BUDGET,  # same cap used for the LangChain agent above
    )
    return VacationPlan.model_validate_json(result.value or "")


print("Mellea session ready:", m)

## Step 11. Run the Mellea agent

Watch the cell output above the plan: Mellea's logger prints `## ReACT TURN NUMBER N` for every turn of the loop by default, no `verbose=True` flag needed.

In [ ]:
mellea_plan = await run_mellea_agent(vacation_request)

print("=== MELLEA AGENT PLAN ===")
print(mellea_plan.model_dump_json(indent=2))

## Step 12. A Requirement-checked, auto-repaired synthesis

A `Requirement` is a rule attached to a Mellea call, checked automatically after the model responds. `react(format=...)` (Step 10) bundles the tool loop and the final synthesis into one call, with no room to attach one. Splitting synthesis into its own `instruct()` call unlocks Mellea's real instruct-validate-repair loop (see [Tutorial 04](https://github.com/generative-computing/mellea/blob/main/docs/docs/tutorials/04-making-agents-reliable.md)):

- `budget_requirement` wraps `budget_check` in a `Requirement`, via a `validation_fn` that parses the JSON output. ([Write Custom Verifiers](https://github.com/generative-computing/mellea/blob/main/docs/docs/how-to/write-custom-verifiers.md), [Enforce Structured Output](https://github.com/generative-computing/mellea/blob/main/docs/docs/how-to/enforce-structured-output.md#validating-structured-output-content))
- `RepairTemplateStrategy` feeds the failure *reason* back into the next attempt's prompt, unlike `RejectionSamplingStrategy`, which just resamples blind. Step 8's LangChain retry has no equivalent.
- A fresh discovery `react()` call gathers the facts again, since `m.ctx` needs the tool trail before this synthesis step.

In [ ]:
def validate_budget(ctx) -> ValidationResult:
    """Requirement validator: parse the JSON output and reuse budget_check."""
    output = ctx.last_output()
    try:
        plan = VacationPlan.model_validate_json(output.value or "")
    except Exception as exc:
        return ValidationResult(False, reason=f"Output did not match the VacationPlan schema: {exc}")
    ok, reason = budget_check(plan)
    return ValidationResult(ok, reason=None if ok else reason)


budget_requirement = Requirement(
    "The itinerary must cover every requested day, and the estimated total cost "
    "must not exceed the traveler's budget.",
    validation_fn=validate_budget,
)

_, discovery_ctx = await react(
    goal=vacation_request,
    context=ChatContext(),
    backend=m.backend,
    tools=[mellea_destination_tool],
    loop_budget=TOOL_CALL_BUDGET,
)

m.ctx = discovery_ctx
sampling = m.instruct(
    "Using the destination facts you already gathered, write the final vacation plan.",
    format=VacationPlan,
    requirements=[budget_requirement],
    strategy=RepairTemplateStrategy(loop_budget=3),
    return_sampling_results=True,
    tool_calls=False,
)

print("=== VALIDATING MELLEA AGENT ===")
for i, validations in enumerate(sampling.sample_validations, start=1):
    for requirement, val_result in validations:
        status = "PASS" if val_result else "FAIL"
        print(f"  Attempt {i}: {status} - {val_result.reason or requirement.description}")

validated_mellea_plan = VacationPlan.model_validate_json(str(sampling.result))
print(validated_mellea_plan.model_dump_json(indent=2))

### Side-by-side: LangChain agent vs. Mellea agent

| | LangChain Agent | Mellea Agent |
|---|---|---|
| **Setup** | `ChatOllama` + `create_agent(model, tools, system_prompt, middleware)` | `start_session()` + `react(goal, context, backend, tools, format, loop_budget)` |
| **Tool definition** | `destination_guide` decorated with `@tool` | Same object, wrapped with `MelleaTool.from_langchain()`, zero rewrite |
| **Output typing** | Needs the provider to support forced tool choice; `ChatOllama` doesn't, so a local model needs a second `with_structured_output()` call | `format=VacationPlan` on `react()` itself: native constrained decoding, no tool-choice dependency |
| **Loop control** | `ToolCallLimitMiddleware(run_limit=...)`, one option from a larger middleware catalog | `loop_budget=...`, a plain keyword argument |
| **Validation** | Hand-written retry wrapper; retries blind, same prompt every time | `Requirement` + `RepairTemplateStrategy`: failure *reason* fed back into the next attempt |
| **Observability** | `debug=True`, or `.stream(..., stream_mode="updates")` | Per-turn `## ReACT TURN NUMBER N` logging by default; `ChatContext` is directly inspectable |

Getting a typed plan wasn't symmetric: one call for Mellea, two for LangChain once forced tool choice failed on Ollama. Validating it wasn't either; only Mellea's repair loop tells the model *why* the last attempt failed.

## Key Takeaways

- `MelleaTool.from_langchain()` wraps a LangChain `BaseTool` unmodified: one tool instance, used as-is by both agents.
- Typed output isn't automatic everywhere: `response_format=` needs forced tool choice, which `ChatOllama` doesn't support (confirmed on `granite4.2:3b`). Mellea's `format=` uses constrained decoding directly, one call instead of two.
- A typed schema guarantees *shape*, not *content*. Both agents still needed a `budget_check` on top of it.
- `react()` has no `requirements=`, so a validated retry is your own code either way, but `Requirement` + `RepairTemplateStrategy` (Step 12) feeds the failure reason back automatically; a hand-rolled retry (Step 8) just tries again blind.
- Loop control: a plain keyword (`loop_budget=`) vs. LangChain's larger middleware catalog (human-in-the-loop, PII redaction, model fallback, and more).
- `Requirement` + repair strategies apply to every `instruct()` call in a Mellea codebase, not just agents. See [Mellea vs Orchestration Frameworks](https://github.com/generative-computing/mellea/blob/main/docs/docs/concepts/architecture-vs-agents.md).

## Next Steps

- [Instruct-Validate-Repair](../InstructValidateRepair/InstructValidateRepair.ipynb): a deeper look at `Requirement`, `RejectionSamplingStrategy`, and the repair loop hand-rolled in Steps 8 and 12 above.
- [Structured Data Extraction](../StructuredDataExtraction/StructuredDataExtraction.ipynb): more on `format=` and building Pydantic schemas for Mellea to fill in.
- [Structured Sentiment Classifier](../SentimentClassifier/SentimentClassifier.ipynb): the same raw-call-vs-Mellea contrast this notebook uses, applied to a single classification call instead of an agent loop.